In [1]:
"""Comparing source transcript and generated transcripts for TTS test files using jiwer metrics."""
import jiwer


def evaluate_text_transformed(reference, hypothesis):
    """Compare transcript strings with the following methods:
    - Word Error Rate (WER)
    - Match Error Rate (MER)
    - Word Information Lost (WIL)
    - Word Information Preserved (WIP)
    - Character Error Rate (CER)
    """
    transformation = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords(),
    ])

    output_words = jiwer.process_words(reference, hypothesis)
    output_chars = jiwer.process_characters(reference, hypothesis)


    output_words_transformed = jiwer.process_words(
        reference,
        hypothesis,
        reference_transform=transformation,
        hypothesis_transform=transformation
    )
    output_chars_transformed = jiwer.process_characters(
        reference,
        hypothesis,
        reference_transform=transformation,
        hypothesis_transform=transformation
    )

    formatted_output = '\n'.join([
        f'{"":<5}{"Default":>12} |{"Custom":>12}',
        f'{"WER:":<5}{output_words.wer:>12.2%} |{output_words_transformed.wer:>12.2%}',
        f'{"MER:":<5}{output_words.mer:>12.2%} |{output_words_transformed.mer:>12.2%}',
        f'{"WIL:":<5}{output_words.wil:>12.2%} |{output_words_transformed.wil:>12.2%}',
        f'{"WIP:":<5}{output_words.wip:>12.2%} |{output_words_transformed.wip:>12.2%}',
        f'{"CER:":<5}{output_chars.cer:>12.2%} |{output_chars_transformed.cer:>12.2%}',
    ])

    return formatted_output


FILENAMES = [
    'dictation01',
    'dictation02',
    'dictation03',
    'dictation04',
    'encounter01',
    'encounter02',
]

# Get source transcript file paths in test directory
SOURCE_TRANSCRIPT_DIR = './samples/tts_transcripts/'

# Get generated transcript file paths in test directory
GENERATED_TRANSCRIPT_DIR = './results/transcripts/'
GENERATED_TRANSCRIPT_SUBDIRS = ['gpt', 'whisper']

# Compare source transcript and generated transcripts for each file
for filename in FILENAMES:
    print(f'Comparing source transcript and generated transcripts for: test_tts_{filename}.mp3')

    # Get text from source transcript file
    source_transcript_path = f'{SOURCE_TRANSCRIPT_DIR}test_gpt_{filename}.txt'
    with open(source_transcript_path, 'r') as source_file:
        source_transcript = source_file.read()

    print('Source transcript: ' + source_transcript_path)
    for subdir in GENERATED_TRANSCRIPT_SUBDIRS:
        method = 'gpt-4o-mini-transcribe' if subdir == 'gpt' else 'faster-whisper-large-v3-turbo'
        print('Method: ' + method)

        # Get text from generated transcript file
        gen_file_path = f'{GENERATED_TRANSCRIPT_DIR}{subdir}/{filename}_{subdir}.txt'
        print('File: ' + gen_file_path + '\n')
        with open(gen_file_path, 'r') as gen_file:
            generated_transcript = gen_file.read()

            # Evaluate the generated transcript against the source transcript
            print('Results with default and custom transformation (lower case, remove punctuation, remove multiple spaces, strip):')
            result = evaluate_text_transformed(source_transcript, generated_transcript)
            print(result, '\n')
    print('--' * 20 + '\n')



Comparing source transcript and generated transcripts for: test_tts_dictation01.mp3
Source transcript: ./samples/tts_transcripts/test_gpt_dictation01.txt
Method: gpt-4o-mini-transcribe
File: ./results/transcripts/gpt/dictation01_gpt.txt

Results with default and custom transformation (lower case, remove punctuation, remove multiple spaces, strip):
          Default |      Custom
WER:       21.30% |      14.08%
MER:       20.07% |      13.27%
WIL:       31.96% |      19.88%
WIP:       68.04% |      80.12%
CER:        5.67% |      14.08% 

Method: faster-whisper-large-v3-turbo
File: ./results/transcripts/whisper/dictation01_whisper.txt

Results with default and custom transformation (lower case, remove punctuation, remove multiple spaces, strip):
          Default |      Custom
WER:       21.66% |      14.08%
MER:       20.41% |      13.27%
WIL:       32.53% |      19.88%
WIP:       67.47% |      80.12%
CER:        6.90% |      14.08% 

----------------------------------------

Comparing